In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("ReadDataFromS3")
    .config("spark.driver.memory", "12g")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.DefaultAWSCredentialsProviderChain",
    )
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000")  # 30s
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000")  # 60s
    .config("spark.hadoop.fs.s3a.connection.timeout", "200000")  # 200s
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400000")  # 24h em ms
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1")
    .getOrCreate()
)


s3_bucket_path = "s3a://aws-etl-pipeline-nyc-taxi/bronze/nyc_taxi/"
df = spark.read.csv(s3_bucket_path, header=True, inferSchema=True)
df.show()

# Define the S3 path (using s3a protocol)
s3_bucket_path = "s3a://aws-etl-pipeline-nyc-taxi/bronze/nyc_taxi/"

# Read CSV data from S3 into a DataFrame
df = spark.read.csv(s3_bucket_path, header=True, inferSchema=True)

# Show data
df.show()

+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|   pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|   -73.993896484375|40.750110626220703|        

+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|   pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|   -73.993896484375|40.750110626220703|        

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
conf = spark._jsc.hadoopConfiguration()

# pegar as keys via Java iterator
it = conf.iterator()
while it.hasNext():
    entry = it.next()
    k = entry.getKey()
    v = entry.getValue()
    if "s3a" in k.lower() and "h" in v.lower():
        print(k, "=", v)


mapreduce.outputcommitter.factory.scheme.s3a = org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory
fs.s3a.impl = org.apache.hadoop.fs.s3a.S3AFileSystem
fs.s3a.buffer.dir = ${env.LOCAL_DIRS:-${hadoop.tmp.dir}}/s3a
fs.viewfs.overload.scheme.target.s3a.impl = org.apache.hadoop.fs.s3a.S3AFileSystem
fs.s3a.aws.credentials.provider = com.amazonaws.auth.DefaultAWSCredentialsProviderChain
fs.s3a.assumed.role.credentials.provider = org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider
fs.AbstractFileSystem.s3a.impl = org.apache.hadoop.fs.s3a.S3A


In [5]:
df.printSchema()

# Mostra o número de registros lidos
print(f"Total de registros lidos de todos os arquivos: {df.count()}")

# Mostra as primeiras 20 linhas do DataFrame
df.show()


root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)



Total de registros lidos de todos os arquivos: 47248845
+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|   pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|     

In [6]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, TimestampType

print("Iniciando a conversão de tipos de dados...")

df_typed = df \
    .withColumn("VendorID", col("VendorID").cast(IntegerType())) \
    .withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast(TimestampType())) \
    .withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast(TimestampType())) \
    .withColumn("passenger_count", col("passenger_count").cast(IntegerType())) \
    .withColumn("trip_distance", col("trip_distance").cast(DoubleType())) \
    .withColumn("RateCodeID", col("RateCodeID").cast(IntegerType())) \
    .withColumn("payment_type", col("payment_type").cast(IntegerType())) \
    .withColumn("fare_amount", col("fare_amount").cast(DoubleType())) \
    .withColumn("extra", col("extra").cast(DoubleType())) \
    .withColumn("mta_tax", col("mta_tax").cast(DoubleType())) \
    .withColumn("tip_amount", col("tip_amount").cast(DoubleType())) \
    .withColumn("tolls_amount", col("tolls_amount").cast(DoubleType())) \
    .withColumn("improvement_surcharge", col("improvement_surcharge").cast(DoubleType())) \
    .withColumn("total_amount", col("total_amount").cast(DoubleType())) \
    .withColumn("pickup_longitude", col("pickup_longitude").cast(DoubleType())) \
    .withColumn("pickup_latitude", col("pickup_latitude").cast(DoubleType())) \
    .withColumn("dropoff_longitude", col("dropoff_longitude").cast(DoubleType())) \
    .withColumn("dropoff_latitude", col("dropoff_latitude").cast(DoubleType()))

print("Conversão de tipos concluída. Verificando o novo schema:")

# Ação para verificar o resultado
df_typed.printSchema()

Iniciando a conversão de tipos de dados...
Conversão de tipos concluída. Verificando o novo schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [7]:
from pyspark.sql.functions import col, hour, when
from pyspark.sql.types import BooleanType

# Supondo que 'df_cleaned' é seu DataFrame após a limpeza inicial

print("Iniciando o enriquecimento do DataFrame com padrões em inglês...")

# Primeiro, criamos as colunas de duração e período do dia com os novos nomes
df_enriched = df_typed.withColumn(
    "trip_duration_minutes",
    (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60
).withColumn(
    "day_part",
    when((hour(col("tpep_pickup_datetime")) >= 6) & (hour(col("tpep_pickup_datetime")) < 12), "morning")
    .when((hour(col("tpep_pickup_datetime")) >= 12) & (hour(col("tpep_pickup_datetime")) < 18), "afternoon")
    .when((hour(col("tpep_pickup_datetime")) >= 18) & (hour(col("tpep_pickup_datetime")) < 23), "evening")
    .otherwise("late_night")
)

# Agora, criamos a coluna 'is_valid' como um booleano.
# A lógica aqui é definir a condição para uma viagem ser VÁLIDA.
is_valid_condition = (
    (col("trip_duration_minutes") > 0) &
    (col("passenger_count") > 0) &
    (col("total_amount") > 0) &
    (col("trip_distance") > 0)
)

df_final_enriched = df_enriched.withColumn("is_valid", is_valid_condition)


print("Enriquecimento finalizado. Verificando resultado com o novo schema:")
df_final_enriched.printSchema()

print("\nAmostra dos novos dados:")
df_final_enriched.select("trip_duration_minutes", "day_part", "is_valid").show(10)

print("\nContagem de registros válidos vs. inválidos:")
df_final_enriched.groupBy("is_valid").count().show()

print(df_final_enriched.printSchema())

Iniciando o enriquecimento do DataFrame com padrões em inglês...
Enriquecimento finalizado. Verificando resultado com o novo schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nu

+---------------------+--------+--------+
|trip_duration_minutes|day_part|is_valid|
+---------------------+--------+--------+
|                18.05| evening|    true|
|   19.833333333333332| evening|    true|
|                10.05| evening|    true|
|   1.8666666666666667| evening|    true|
|   19.316666666666666| evening|    true|
|   20.216666666666665| evening|    true|
|   24.866666666666667| evening|    true|
|    8.683333333333334| evening|    true|
|    37.93333333333333| evening|    true|
|    7.066666666666666| evening|    true|
+---------------------+--------+--------+
only showing top 10 rows

Contagem de registros válidos vs. inválidos:


+--------+--------+
|is_valid|   count|
+--------+--------+
|    true|46944037|
|   false|  304808|
+--------+--------+

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = tru

In [8]:
from pyspark.sql.functions import col


print("Iniciando processo de limpeza de dados...")

# --- Passo 0: Contagem Inicial ---
initial_count = df_final_enriched.count()
print(f"Contagem inicial de registros: {initial_count:,}")

# --- Passo 1: Remoção de Linhas Duplicadas ---
# Remove registros que são uma cópia exata um do outro em todas as colunas.
df_deduplicated = df_final_enriched.dropDuplicates()
count_after_deduplication = df_deduplicated.count()
duplicates_removed = initial_count - count_after_deduplication
print(f"Linhas duplicatas removidas: {duplicates_removed:,}")

# --- Passo 2: Remoção de Linhas com Nulos em Colunas Críticas ---
critical_columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "total_amount"]
df_no_nulls = df_deduplicated.dropna(subset=critical_columns)
count_after_null_removal = df_no_nulls.count()
nulls_removed = count_after_deduplication - count_after_null_removal
print(f"Linhas com nulos em colunas críticas removidas: {nulls_removed:,}")

# --- Passo 3: Filtragem por Lógica de Negócio (usando a coluna 'is_valid') ---
# df_cleaned = df_no_nulls.filter(col("is_valid") == True)
# count_after_validation = df_cleaned.count()
# invalid_removed = count_after_null_removal - count_after_validation
# print(f"Linhas inválidas (segundo as regras de negócio) removidas: {invalid_removed:,}")

# Comentar essas linhas se formos remover os dados inválidos
invalid_removed = 0
count_after_validation = df_no_nulls.count()

# --- Passo 4: Resumo Final ---
final_count = count_after_validation
total_removed = initial_count - final_count
percentage_retained = (final_count / initial_count) * 100 if initial_count > 0 else 0

print("\n--- RESUMO DA LIMPEZA ---")
print(f"Contagem Inicial: {initial_count:>15,}")
print(f"  - Duplicatas Removidas: {duplicates_removed:>10,}")
print(f"  - Nulos Removidos: {nulls_removed:>15,}")
print(f"  - Inválidos Removidos: {invalid_removed:>12,}")
print("---------------------------------")
print(f"Contagem Final: {final_count:>17,}")
print(f"Total de Registros Removidos: {total_removed:,}")
print(f"Percentual de Dados Retidos: {percentage_retained:.2f}%")

# O DataFrame 'df_cleaned' está pronto para ser salvo na camada Silver.

Iniciando processo de limpeza de dados...


Contagem inicial de registros: 47,248,845


Linhas duplicatas removidas: 386


Linhas com nulos em colunas críticas removidas: 0



--- RESUMO DA LIMPEZA ---
Contagem Inicial:      47,248,845
  - Duplicatas Removidas:        386
  - Nulos Removidos:               0
  - Inválidos Removidos:            0
---------------------------------
Contagem Final:        47,248,459
Total de Registros Removidos: 386
Percentual de Dados Retidos: 100.00%


In [10]:
from pyspark.sql.functions import year, month

# Supondo que 'df_cleaned' é o seu DataFrame final após todo o processo de limpeza.

print("Iniciando a escrita dos dados na camada Silver...")

# --- Passo 1: Criar as colunas de partição ---
# Adicionamos colunas de 'year' e 'month' baseadas na data de início da viagem.
# Usaremos essas colunas para particionar os dados no S3.
df_for_silver = df_no_nulls.withColumn("year", year(col("tpep_pickup_datetime"))) \
                          .withColumn("month", month(col("tpep_pickup_datetime")))

# --- Passo 2: Definir o caminho de destino no S3 ---
bucket_name = "aws-etl-pipeline-nyc-taxi"
silver_s3_path = f"s3a://{bucket_name}/silver/nyc_taxi_trips/"

# --- Passo 3: Escrever o DataFrame em formato Parquet, particionado ---
(df_for_silver.write
    .mode("overwrite")  # Sobrescreve os dados se a pasta já existir. Cuidado em produção!
    .partitionBy("year", "month")  # A "mágica" da otimização acontece aqui.
    .parquet(silver_s3_path)
)

print("\n-------------------------------------------------------------")
print(f"SUCESSO! Os dados foram escritos na camada Silver em:")
print(f"{silver_s3_path}")
print("-------------------------------------------------------------")

Iniciando a escrita dos dados na camada Silver...



-------------------------------------------------------------
SUCESSO! Os dados foram escritos na camada Silver em:
s3a://aws-etl-pipeline-nyc-taxi/silver/nyc_taxi_trips/
-------------------------------------------------------------
